In [1]:
# Audio extraction
from pathlib import Path
import subprocess

def extract_audio(
    video_path: str,
    output_path: str,
    sample_rate: int = 16000
):
    """
    Extracts audio from a video file using FFmpeg.

    Parameters
    ----------
    video_path : str
        Path to the input video file.
    output_path : str
        Path to save the extracted audio (usually .wav).
    sample_rate : int, optional
        Target sample rate for audio (default 16000 Hz).
    """

    video_path = Path(video_path)
    output_path = Path(output_path)

    # Make sure the parent folder exists
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # FFmpeg command to extract audio
    command = [
        "ffmpeg",
        "-y",                      # overwrite output if it exists
        "-i", str(video_path),      # input file
        "-vn",                     # ignore video
        "-ac", "1",                # mono audio
        "-ar", str(sample_rate),    # sample rate
        str(output_path)            # output file
    ]

    subprocess.run(command, check=True)    

## Extract audio
video_path = "../data/raw/example1.mp4"
audio_path = "../data/processed/example1.wav"

## For notebook only
if (not Path(audio_path).exists()):
    extract_audio(video_path, audio_path)
    print("Sound extracted!")
else:
    print("Sound has already been extracted previously!")


Sound has already been extracted previously!


In [2]:
import whisper
import time

model_to_use = "small"
print(f"Processing model: {model_to_use}")

model = whisper.load_model(model_to_use)
start_time = time.time()
result = model.transcribe(audio_path, word_timestamps=True)
end_time = time.time()
print(f"Time: {end_time - start_time}")
print(result['text'])
################ Findings ###################
# Note: I do not have an official copy of the transcript for the videos
# so i cannot measure accuracy easily.
## Medium and large simply take too much RAM on my computer to be feasible

#### Example 1: tommyinnit swearing ####
# Video notes: Generally the audio itself is clear, and the people speak clearly, so i suspect that
# the accuracy will be pretty good. However, speakers have different accents (British and American)
# and talk over each other frequently, so transcription might not be perfect.
# Video length - 4:05

## Tiny: 26.0 seconds. Accuracy at first glance seems great, but definitely has errors
## Base: 26.5 seconds. About the same as tiny.
## Small: 106 seconds. Seems pretty much perfect bar one or two minor errors

# Small is 4x slower than tiny but it is not tooo slow. For now, I will stick with tiny for testing purposes.

Processing model: small


c:\Users\anant\AppData\Local\Programs\Python\Python311\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Time: 126.84636855125427
 I swore so much at a man that hates swearing. How angry can I make bad boy Halo before I make everyone upset? He completely lost his mind. Also only a small percentage of people who watch my videos actually click the subscribe button. So if you like this then please subscribe. It's 100% free and you can always unsubscribe later. Enjoy! Hey, bad. Hi! Yeah, I saw you at a call, but I wasn't at my computer. Where were you? Playing fetch with rat. Is your dog really named rat? No, that's just a nickname. Anyway, what's up? What did you call? Do you want to know the nickname for my dog? Sure. Bitch. Language Tommy. Why is it with you in language bad? Why is it? Why is it with you and swearing? What happened? Is it like some childhood trauma where like someone sweared at you and then you're like someone died or like what is it? Like I just like to keep content that I'm in family friendly. My content is not for family nor friends. It's not for family or friends? No. 

In [3]:
# load profanity list
profanity_set = set()
profanity_file_path = "../src/content_filter/config/profanity_words.txt"
with open(profanity_file_path, "r", encoding="utf-8") as f:
    for line in f:
        word = line.strip().lower()
        profanity_set.add(word)

In [5]:
import re

def remove_punctuation(text: str) -> str:
    return re.sub(r"[^\w\s]", "", text)

ans = []
for seg in result['segments']:
    # Note to self: will likely generate false positives as i am not looking at probability.
    for wordStamp in seg['words']:
        word = remove_punctuation(wordStamp['word'].strip()).lower()
        if word in profanity_set:
            ans.append(wordStamp)

print(ans)

[{'word': ' Bitch.', 'start': np.float64(41.88), 'end': np.float64(42.2), 'probability': np.float64(0.119268037378788)}, {'word': ' shit,', 'start': np.float64(78.6), 'end': np.float64(78.96), 'probability': np.float64(0.6835004687309265)}, {'word': ' Wanker,', 'start': np.float64(79.83999999999999), 'end': np.float64(80.32), 'probability': np.float64(0.640914024785161)}, {'word': ' Penis', 'start': np.float64(81.06), 'end': np.float64(81.46), 'probability': np.float64(0.6247397512197495)}, {'word': ' piss', 'start': np.float64(102.96), 'end': np.float64(103.04), 'probability': np.float64(0.9366893768310547)}, {'word': ' Piss', 'start': np.float64(148.36), 'end': np.float64(148.64), 'probability': np.float64(0.11842300929129124)}, {'word': ' Piss', 'start': np.float64(149.6), 'end': np.float64(149.68), 'probability': np.float64(0.9363669753074646)}]
